# 🩻🧠 RSNA Knee — Report-Guided Multi-Series 2.5D GPU Baseline v3

## Strategic summary

**Primary objective:** recover a real MRI signal above the previous ~0.51 AUC regime while keeping the end-to-end Tesla T4 runtime **below two hours**.

The previous run established three facts: (1) DICOM coverage is complete, (2) random initialization alone is not a defensible representation strategy for this small/sparse supervision regime, and (3) the old 12-token / 12-epoch schedule is too expensive. This version changes the representation and the runtime budget at the same time.

### v3 design

| Component | v3 decision | Why it should help |
|---|---|---|
| Runtime | **160 px, 6 MRI tokens/study, max 4 supervised epochs** | Roughly halves image-forward volume versus the 12-token baseline |
| Preprocessing | **Direct competition path + adaptive slice ordering** | Avoids expensive recursive root discovery and full header sorting when filenames are already ordered |
| Initialization | **Optional ImageNet checkpoint, never required** | Uses pretraining when available without introducing a hard dependency |
| Multimodal signal | **Leakage-safe report-semantic warm-up** | Uses paired radiology reports as privileged train-only supervision without requiring reports at test time |
| Sparse labels | **Target-aware oversampling + capped positive weighting** | Gives rare positive studies enough gradient exposure |
| Numerical safety | **Finite-value MRI sanitization + fail-fast non-finite loss checks** | Prevents silent NaN training |
| Validation | **Prevalence-stable holdout, raw macro ROC AUC** | Leaderboard movement is not accepted without validation support |

### Training protocol

1. Fit TF-IDF + TruncatedSVD **only on training-split reports**.
2. Warm up the image representation for one short epoch by predicting the dense report-semantic embedding.
3. Fine-tune the MRI model on the 12 official targets for at most four epochs, retaining a small report-semantic auxiliary loss.
4. Select the best checkpoint strictly by validation macro ROC AUC.
5. Stop training early if the wall-clock guard indicates that the two-hour budget is at risk.

### Acceptance gate

- **Reject:** macro AUC < 0.58 or non-finite training.
- **Promising baseline:** macro AUC ≥ 0.62.
- **Strong single-fold baseline:** macro AUC ≥ 0.68.

This notebook does not fabricate labels, overwrite official supervision, smooth predictions toward priors, or use public-LB feedback inside training.

## 1. Imports, reproducibility and hard runtime budget

In [ ]:
import os, re, gc, math, time, random, warnings
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models import resnet18

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

SEED = 2026
IMG_SIZE = 160
BATCH_SIZE = 8
TRIPLETS_PER_PLANE = 2
PLANES = ('sagittal', 'coronal', 'axial')
N_TOKENS = len(PLANES) * TRIPLETS_PER_PLANE
REPORT_DIM = 32
REPORT_WARMUP_EPOCHS = 1
SUPERVISED_EPOCHS = 4
WARMUP_MAX_STUDIES = 3000
HEAD_LR = 3e-4
ENCODER_LR_PRETRAINED = 5e-5
ENCODER_LR_RANDOM = 2e-4
REPORT_AUX_WEIGHT = 0.10
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.15
MAX_POS_WEIGHT = 3.0
EARLY_STOPPING_PATIENCE = 2
NUM_WORKERS = min(4, os.cpu_count() or 2)
INDEX_WORKERS = min(12, max(4, os.cpu_count() or 4))
MAX_RUNTIME_SECONDS = 2 * 60 * 60
TRAINING_CUTOFF_SECONDS = 92 * 60  # preserve time for validation/inference/export

START_TIME = time.time()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'


def elapsed_min():
    return (time.time() - START_TIME) / 60.0


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()
torch.set_float32_matmul_precision('high')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('CPU workers:', NUM_WORKERS, '| index workers:', INDEX_WORKERS)
print('Hard runtime budget: < 120 minutes')

## 2. Fast competition discovery and schema

In [ ]:
SUBMISSION_ID = 'StudyInstanceUID'
KNOWN_ROOT = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')


def find_root():
    if (KNOWN_ROOT / 'sample_submission.csv').exists():
        return KNOWN_ROOT
    # Cheap fallback: inspect only shallow Kaggle input directories.
    base = Path('/kaggle/input')
    candidates = []
    for child in base.iterdir():
        if not child.is_dir():
            continue
        direct = child / 'sample_submission.csv'
        if direct.exists():
            candidates.append(child)
        for sub in list(child.iterdir())[:100]:
            if sub.is_dir() and (sub / 'sample_submission.csv').exists():
                candidates.append(sub)
    for root in candidates:
        try:
            s = pd.read_csv(root / 'sample_submission.csv', nrows=2)
            if SUBMISSION_ID in s.columns and len(s.columns) == 13:
                return root
        except Exception:
            pass
    raise FileNotFoundError('RSNA Knee competition data not found.')

COMP_ROOT = find_root()
sample_submission = pd.read_csv(COMP_ROOT / 'sample_submission.csv')
TARGETS = [c for c in sample_submission.columns if c != SUBMISSION_ID]

train_path = None
test_path = None
for p in COMP_ROOT.glob('*.csv'):
    if p.name == 'sample_submission.csv':
        continue
    x = pd.read_csv(p, nrows=3)
    if SUBMISSION_ID not in x.columns:
        continue
    if set(TARGETS).issubset(x.columns):
        train_path = p
    elif len(x.columns) <= 3:
        test_path = p
if train_path is None or test_path is None:
    raise FileNotFoundError('Could not identify train.csv / test.csv.')

train_raw = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print('Competition root:', COMP_ROOT)
print('Train:', train_path.name, train_raw.shape)
print('Test:', test_path.name, test_df.shape)
print('Targets:', TARGETS)
print('Extra train columns:', [c for c in train_raw.columns if c not in [SUBMISSION_ID] + TARGETS])
print(f'Elapsed: {elapsed_min():.2f} min')

## 3. Strict label audit

Raw supervision is preserved. Unknown encodings fail fast instead of being silently coerced.

In [ ]:
TRUE_TOKENS = {'1','true','t','yes','y','positive','present','abnormal'}
FALSE_TOKENS = {'0','false','f','no','n','negative','absent','normal'}
MISSING_TOKENS = {'','nan','none','null','na','n/a','unknown'}


def decode_binary_series(s, name):
    if pd.api.types.is_bool_dtype(s):
        return s.astype(float)
    if pd.api.types.is_numeric_dtype(s):
        vals = set(pd.Series(s.dropna().unique()).astype(float).tolist())
        if vals.issubset({0.0, 1.0}):
            return s.astype(float)
        raise ValueError(f'{name}: non-binary numeric values: {sorted(vals)[:20]}')
    out=[]; unknown=Counter()
    for v in s:
        if pd.isna(v):
            out.append(np.nan); continue
        token = str(v).strip().lower()
        if token in TRUE_TOKENS: out.append(1.0)
        elif token in FALSE_TOKENS: out.append(0.0)
        elif token in MISSING_TOKENS: out.append(np.nan)
        else:
            try:
                fv=float(token)
                if fv in (0.0,1.0): out.append(fv)
                else: unknown[token]+=1; out.append(np.nan)
            except Exception:
                unknown[token]+=1; out.append(np.nan)
    if unknown:
        raise ValueError(f'{name}: unsupported tokens: {unknown.most_common(10)}')
    return pd.Series(out, index=s.index, dtype=float)

train_df = train_raw.copy()
for t in TARGETS:
    train_df[t] = decode_binary_series(train_raw[t], t)

label_summary = pd.DataFrame({
    'positive': (train_df[TARGETS] == 1).sum(),
    'negative': (train_df[TARGETS] == 0).sum(),
    'missing': train_df[TARGETS].isna().sum(),
    'positive_rate_observed': train_df[TARGETS].mean(),
})
label_summary['missing_rate'] = label_summary['missing'] / len(train_df)
display(label_summary)

TOTAL_POS = int(label_summary['positive'].sum())
print('Total decoded positive labels:', TOTAL_POS)
print('Mean positive labels/study:', round(float(train_df[TARGETS].fillna(0).sum(axis=1).mean()), 4))
if TOTAL_POS < 1000:
    print('⚠️ Sparse-target regime confirmed. v3 activates report-guided warm-up and target-aware oversampling.')
if ((label_summary['positive'] < 2) | (label_summary['negative'] < 2)).any():
    raise RuntimeError('At least one target cannot support ROC AUC.')

## 4. Label visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
label_summary['positive_rate_observed'].sort_values(ascending=False).plot(kind='bar', ax=ax)
ax.set_title('Observed positive prevalence by target')
ax.set_ylabel('Positive fraction among observed labels')
ax.grid(axis='y', alpha=.25)
plt.xticks(rotation=40, ha='right'); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(12,4))
label_summary['missing_rate'].sort_values(ascending=False).plot(kind='bar', ax=ax)
ax.set_title('Missing-label rate by target')
ax.set_ylabel('Missing fraction')
ax.grid(axis='y', alpha=.25)
plt.xticks(rotation=40, ha='right'); plt.tight_layout(); plt.show()

## 5. Leakage-safe prevalence-stable holdout

In [ ]:
PATIENT_CANDIDATES = ['PatientID','patient_id','PatientId','patientId','SubjectID','subject_id']
patient_col = next((c for c in PATIENT_CANDIDATES if c in train_df.columns), None)


def split_score(frame, tr_idx, va_idx):
    tr, va = frame.iloc[tr_idx][TARGETS], frame.iloc[va_idx][TARGETS]
    drift = (tr.mean() - va.mean()).abs().fillna(1.0)
    penalty=0.0
    for t in TARGETS:
        obs=va[t].dropna()
        if obs.nunique() < 2: penalty += 20.0
        elif (obs==1).sum() < 3: penalty += 2.0
    return float(drift.mean() + .5*drift.max() + penalty)


def choose_holdout(frame, n_candidates=100):
    idx=np.arange(len(frame)); cands=[]
    if patient_col is not None and frame[patient_col].nunique(dropna=True) < len(frame):
        groups=frame[patient_col].fillna(frame[SUBMISSION_ID]).astype(str)
        for seed in range(SEED, SEED+n_candidates):
            sp=GroupShuffleSplit(n_splits=1,test_size=VAL_FRACTION,random_state=seed)
            tr,va=next(sp.split(idx,groups=groups)); cands.append((split_score(frame,tr,va),tr,va,f'grouped:{patient_col}'))
    else:
        for seed in range(SEED, SEED+n_candidates):
            sp=ShuffleSplit(n_splits=1,test_size=VAL_FRACTION,random_state=seed)
            tr,va=next(sp.split(idx)); cands.append((split_score(frame,tr,va),tr,va,'study-level optimized holdout'))
    return min(cands,key=lambda z:z[0])

split_quality, train_idx, val_idx, split_method = choose_holdout(train_df)
train_part = train_df.iloc[train_idx].reset_index(drop=True)
val_part = train_df.iloc[val_idx].reset_index(drop=True)
print('Split:', split_method, '| train/val:', len(train_part), len(val_part), '| score:', round(split_quality,6))

split_diag=pd.DataFrame({
    'train_pos': (train_part[TARGETS]==1).sum(),
    'val_pos': (val_part[TARGETS]==1).sum(),
    'train_prev': train_part[TARGETS].mean(),
    'val_prev': val_part[TARGETS].mean(),
})
split_diag['abs_drift']=(split_diag.train_prev-split_diag.val_prev).abs()
display(split_diag.round(4))
if any(val_part[t].dropna().nunique()<2 for t in TARGETS):
    raise RuntimeError('Validation contains a single-class target.')

## 6. Train-only report-semantic teacher

The reports are **privileged training information**. TF-IDF and SVD are fitted only on the training split, so the validation images and validation reports never influence representation learning.

In [ ]:
if 'Report' not in train_part.columns:
    raise RuntimeError('Report column not found; v3 requires the paired train radiology reports.')

train_reports = train_part['Report'].fillna('').astype(str).values
val_reports = val_part['Report'].fillna('').astype(str).values

tfidf = TfidfVectorizer(
    lowercase=True, strip_accents='unicode', ngram_range=(1,2),
    min_df=2, max_df=.995, max_features=30000, sublinear_tf=True
)
Xtr_text = tfidf.fit_transform(train_reports)
Xva_text = tfidf.transform(val_reports)
svd = TruncatedSVD(n_components=REPORT_DIM, random_state=SEED)
Ztr = normalize(svd.fit_transform(Xtr_text)).astype(np.float32)
Zva = normalize(svd.transform(Xva_text)).astype(np.float32)

REPORT_EMBED = {}
for uid,z in zip(train_part[SUBMISSION_ID].astype(str), Ztr): REPORT_EMBED[uid]=z
for uid,z in zip(val_part[SUBMISSION_ID].astype(str), Zva): REPORT_EMBED[uid]=z

print('TF-IDF shape:', Xtr_text.shape)
print('Report SVD explained variance:', round(float(svd.explained_variance_ratio_.sum()),4))
print('Report embedding dim:', REPORT_DIM)

fig, ax = plt.subplots(figsize=(8,3.5))
ax.plot(np.cumsum(svd.explained_variance_ratio_), marker='o', markersize=3)
ax.set_title('Cumulative explained variance of train-only report SVD')
ax.set_xlabel('Component'); ax.set_ylabel('Cumulative variance'); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

## 7. Fast path-level DICOM index

In [ ]:
ALL_UIDS = set(train_df[SUBMISSION_ID].astype(str)) | set(test_df[SUBMISSION_ID].astype(str))


def infer_uid(parts):
    for part in reversed(parts):
        if part in ALL_UIDS: return part
    return None

series_files=defaultdict(lambda: defaultdict(list))
scanned=0; fallback_dirs=0
for dirpath,_,filenames in os.walk(COMP_ROOT):
    dcm=[f for f in filenames if f.lower().endswith('.dcm')]
    if not dcm: continue
    uid=infer_uid(Path(dirpath).parts)
    if uid is None:
        try:
            probe=str(Path(dirpath)/dcm[0])
            ds=pydicom.dcmread(probe,stop_before_pixels=True,force=True,specific_tags=['StudyInstanceUID'])
            cand=str(getattr(ds,'StudyInstanceUID',''))
            uid=cand if cand in ALL_UIDS else None
            fallback_dirs += 1
        except Exception: uid=None
    if uid is None: continue
    files=[str(Path(dirpath)/f) for f in dcm]
    series_files[uid][str(dirpath)].extend(files); scanned += len(files)

train_cov=np.mean(train_df[SUBMISSION_ID].astype(str).isin(series_files))
test_cov=np.mean(test_df[SUBMISSION_ID].astype(str).isin(series_files))
print(f'Path-indexed {scanned:,} DICOM files across {len(series_files):,} studies.')
print('Train/test coverage:', f'{train_cov:.2%}', f'{test_cov:.2%}')
print(f'Elapsed: {elapsed_min():.2f} min')
if train_cov < .95 or test_cov < .95: raise RuntimeError('Unexpectedly low DICOM coverage.')

## 8. One-header series metadata and sequence-aware selection

In [ ]:
def classify_plane(iop):
    try:
        x=np.asarray([float(v) for v in iop],dtype=float)
        normal=np.abs(np.cross(x[:3],x[3:6])); axis=int(np.argmax(normal))
        return ('sagittal','coronal','axial')[axis]
    except Exception: return 'unknown'


def sequence_family(desc):
    d=(' '+(desc or '').lower().replace('_',' ')+' ')
    fat=any(k in d for k in ['fat sat','fatsat','fat-sat',' fs ','stir'])
    if re.search(r'\bpd\b',d) or 'proton' in d: base='PD'
    elif re.search(r'\bt2\b',d): base='T2'
    elif re.search(r'\bt1\b',d): base='T1'
    elif 'stir' in d: base='STIR'
    else: base='OTHER'
    return base + ('_FS' if fat and base!='STIR' else '')


def read_meta(item):
    uid,sdir,files=item; probe=files[len(files)//2]
    try:
        ds=pydicom.dcmread(probe,stop_before_pixels=True,force=True,specific_tags=['ImageOrientationPatient','SeriesDescription','ProtocolName','SequenceName'])
        desc=str(getattr(ds,'SeriesDescription','') or getattr(ds,'ProtocolName','') or getattr(ds,'SequenceName',''))
        return uid, {'dir':sdir,'files_unsorted':files,'n_slices':len(files),'plane':classify_plane(getattr(ds,'ImageOrientationPatient',None)),'description':desc,'sequence':sequence_family(desc)}
    except Exception:
        return uid, {'dir':sdir,'files_unsorted':files,'n_slices':len(files),'plane':'unknown','description':'','sequence':'OTHER'}

items=[(uid,sdir,files) for uid,sd in series_files.items() for sdir,files in sd.items()]
SERIES_META=defaultdict(list)
with ThreadPoolExecutor(max_workers=INDEX_WORKERS) as ex:
    futs=[ex.submit(read_meta,it) for it in items]
    for i,f in enumerate(as_completed(futs),1):
        uid,m=f.result(); SERIES_META[uid].append(m)
        if i%5000==0: print(f'Read representative headers: {i:,}/{len(items):,}')

print('Plane counts:', Counter(m['plane'] for v in SERIES_META.values() for m in v))
print('Sequence families:', Counter(m['sequence'] for v in SERIES_META.values() for m in v))

In [ ]:
PLANE_SEQUENCE_PRIORITY={
    'sagittal':['PD_FS','T2_FS','STIR','PD','T2','T1','OTHER'],
    'coronal':['PD_FS','T2_FS','STIR','PD','T2','T1','OTHER'],
    'axial':['T2_FS','PD_FS','STIR','T2','PD','T1','OTHER'],
}

def rank_series(m,plane):
    pref=PLANE_SEQUENCE_PRIORITY[plane]
    try:r=pref.index(m['sequence'])
    except ValueError:r=len(pref)
    return (r,-m['n_slices'])

def choose(uid):
    metas=SERIES_META.get(str(uid),[]); out={}; used=set()
    for plane in PLANES:
        cand=[m for m in metas if m['plane']==plane]
        out[plane]=min(cand,key=lambda m:rank_series(m,plane)) if cand else None
        if out[plane] is not None: used.add(out[plane]['dir'])
    fallback=sorted([m for m in metas if m['dir'] not in used],key=lambda m:-m['n_slices'])
    for plane in PLANES:
        if out[plane] is None and fallback: out[plane]=fallback.pop(0)
    return out

SELECTED={uid:choose(uid) for uid in ALL_UIDS}
rows=[]
for uid,d in SELECTED.items():
    for plane,m in d.items(): rows.append({'uid':uid,'slot':plane,'sequence':None if m is None else m['sequence'],'n_slices':0 if m is None else m['n_slices']})
selected_df=pd.DataFrame(rows)
display(pd.crosstab(selected_df['slot'],selected_df['sequence'],dropna=False))

## 9. Adaptive slice ordering

Full per-slice header sorting was one of the largest runtime costs. v3 first tests whether natural filename order is consistent with DICOM `InstanceNumber` using five probes. Only inconsistent series fall back to a full header sort.

In [ ]:
def natural_key(path):
    name=Path(path).name
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)',name)]


def read_instance(path):
    try:
        ds=pydicom.dcmread(path,stop_before_pixels=True,force=True,specific_tags=['InstanceNumber'])
        v=getattr(ds,'InstanceNumber',None)
        return float(v) if v is not None else np.nan
    except Exception: return np.nan


def adaptive_order(meta):
    if meta is None: return None, 'missing'
    files=sorted(meta['files_unsorted'],key=natural_key)
    n=len(files)
    if n<=2:
        out=dict(meta); out['files']=files; return out,'natural'
    probe_idx=sorted(set([0,n//4,n//2,(3*n)//4,n-1]))
    inst=np.array([read_instance(files[i]) for i in probe_idx],dtype=float)
    finite=np.isfinite(inst)
    if finite.sum()>=3:
        x=inst[finite]
        inc=np.all(np.diff(x)>0); dec=np.all(np.diff(x)<0)
        if inc or dec:
            if dec: files=list(reversed(files))
            out=dict(meta); out['files']=files; return out,'natural_verified'
    # Full fallback only for series whose natural ordering cannot be verified.
    vals=[(p,read_instance(p)) for p in files]
    good=sum(np.isfinite(v) for _,v in vals)
    if good>=max(2,n//2):
        vals.sort(key=lambda z:(np.inf if not np.isfinite(z[1]) else z[1],natural_key(z[0])))
        files=[p for p,_ in vals]; mode='full_instance_sort'
    else:
        mode='natural_unverified'
    out=dict(meta); out['files']=files; return out,mode

unique={}
for uid,d in SELECTED.items():
    for m in d.values():
        if m is not None: unique[m['dir']]=m

ORDERED={}; order_modes=Counter()
with ThreadPoolExecutor(max_workers=INDEX_WORKERS) as ex:
    futs={ex.submit(adaptive_order,m):d for d,m in unique.items()}
    for i,f in enumerate(as_completed(futs),1):
        m,mode=f.result(); ORDERED[m['dir']]=m; order_modes[mode]+=1
        if i%3000==0: print(f'Ordered {i:,}/{len(futs):,} selected series')
for uid,d in SELECTED.items():
    for plane,m in list(d.items()):
        if m is not None: d[plane]=ORDERED[m['dir']]

print('Ordering modes:', order_modes)
print('Selected series:', len(ORDERED))
print(f'Elapsed: {elapsed_min():.2f} min')

## 10. MRI preprocessing sanity check

In [ ]:
def read_dicom_array(path):
    ds=pydicom.dcmread(path,force=True)
    arr=ds.pixel_array.astype(np.float32)
    if arr.ndim>2: arr=arr[arr.shape[0]//2]
    slope=float(getattr(ds,'RescaleSlope',1.0)); intercept=float(getattr(ds,'RescaleIntercept',0.0))
    arr=arr*slope+intercept
    finite=arr[np.isfinite(arr)]
    if finite.size==0: return np.zeros_like(arr,dtype=np.float32)
    lo,hi=np.percentile(finite,(1,99))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi<=lo: return np.zeros_like(arr,dtype=np.float32)
    # Explicit sanitization: corrupted non-finite pixels are mapped to the robust lower/upper window bounds.
    arr=np.nan_to_num(arr,nan=lo,posinf=hi,neginf=lo)
    if str(getattr(ds,'PhotometricInterpretation','')).upper()=='MONOCHROME1': arr=arr.max()-arr
    arr=np.clip(arr,lo,hi); arr=(arr-lo)/(hi-lo)
    return arr.astype(np.float32)

available=[u for u in train_df[SUBMISSION_ID].astype(str) if u in SELECTED]
rng=np.random.default_rng(SEED); examples=rng.choice(available,size=min(3,len(available)),replace=False)
fig,axes=plt.subplots(len(examples),3,figsize=(11,3.4*len(examples))); axes=np.atleast_2d(axes)
for r,uid in enumerate(examples):
    for c,plane in enumerate(PLANES):
        ax=axes[r,c]; m=SELECTED[uid].get(plane)
        if m is None or not m.get('files'): ax.text(.5,.5,'missing',ha='center'); ax.axis('off'); continue
        try: ax.imshow(read_dicom_array(m['files'][len(m['files'])//2]),cmap='gray')
        except Exception as e: ax.text(.5,.5,str(e)[:50],ha='center')
        ax.set_title(f'{plane} | {m["sequence"]} | n={m["n_slices"]}'); ax.axis('off')
plt.suptitle('Selected MRI series sanity check',y=1.01); plt.tight_layout(); plt.show()

## 11. Six-token 2.5D MRI dataset

Each plane contributes two distributed triplets. This is the main runtime reduction: **6 tokens/study instead of 12**.

In [ ]:
INPUT_MEAN=(0.5,0.5,0.5); INPUT_STD=(0.25,0.25,0.25)

def resize2d(arr):
    t=torch.from_numpy(arr).float()[None,None]
    return F.interpolate(t,size=(IMG_SIZE,IMG_SIZE),mode='bilinear',align_corners=False)[0,0]

def centers(n):
    if n<=1:return [0]*TRIPLETS_PER_PLANE
    return [int(round(f*(n-1))) for f in np.linspace(.30,.70,TRIPLETS_PER_PLANE)]

def read_triplet(files,center,train_mode=False):
    n=len(files)
    if n==0:return torch.zeros(3,IMG_SIZE,IMG_SIZE)
    if train_mode and n>6:center=int(np.clip(center+random.randint(-1,1),0,n-1))
    delta=max(1,n//64); ids=[max(0,center-delta),center,min(n-1,center+delta)]
    ch=[]
    for i in ids:
        try: ch.append(resize2d(read_dicom_array(files[i])))
        except Exception: ch.append(torch.zeros(IMG_SIZE,IMG_SIZE))
    x=torch.stack(ch)
    x=torch.nan_to_num(x,nan=0.0,posinf=1.0,neginf=0.0).clamp(0,1)
    if train_mode:
        if random.random()<.5:x=torch.flip(x,dims=[2])
        if random.random()<.25:x=x.pow(random.uniform(.88,1.12))
        if random.random()<.25:x=(x*random.uniform(.92,1.08)+random.uniform(-.03,.03)).clamp(0,1)
    mean=torch.tensor(INPUT_MEAN,dtype=x.dtype)[:,None,None]; std=torch.tensor(INPUT_STD,dtype=x.dtype)[:,None,None]
    return (x-mean)/std

class KneeDataset(Dataset):
    def __init__(self,frame,targets=None,train_mode=False,report_embed=None):
        self.frame=frame.reset_index(drop=True); self.targets=targets; self.train_mode=train_mode; self.report_embed=report_embed or {}
    def __len__(self):return len(self.frame)
    def __getitem__(self,idx):
        row=self.frame.iloc[idx]; uid=str(row[SUBMISSION_ID]); study=SELECTED.get(uid,{})
        tokens=[]; masks=[]; plane_ids=[]; pos_ids=[]
        for p,plane in enumerate(PLANES):
            m=study.get(plane)
            if m is None or not m.get('files'):
                for q in range(TRIPLETS_PER_PLANE):
                    tokens.append(torch.zeros(3,IMG_SIZE,IMG_SIZE)); masks.append(0.0); plane_ids.append(p); pos_ids.append(q)
            else:
                for q,ctr in enumerate(centers(len(m['files']))):
                    tokens.append(read_triplet(m['files'],ctr,self.train_mode)); masks.append(1.0); plane_ids.append(p); pos_ids.append(q)
        out={'image':torch.stack(tokens),'token_mask':torch.tensor(masks,dtype=torch.float32),'plane_id':torch.tensor(plane_ids),'position_id':torch.tensor(pos_ids),'uid':uid}
        if self.targets is not None: out['target']=torch.tensor(row[self.targets].to_numpy(dtype=np.float32),dtype=torch.float32)
        if uid in self.report_embed: out['report_embed']=torch.tensor(self.report_embed[uid],dtype=torch.float32)
        return out

## 12. Optional pretrained encoder — zero hard dependencies

In [ ]:
def quick_checkpoint_candidates():
    out=[]
    cache=Path(torch.hub.get_dir())/'checkpoints'/'resnet18-f37072fd.pth'
    if cache.exists(): out.append(cache)
    base=Path('/kaggle/input')
    if base.exists():
        for child in base.iterdir():
            if not child.is_dir(): continue
            for name in ['resnet18-f37072fd.pth','resnet18.pth']:
                p=child/name
                if p.exists(): out.append(p)
                # one additional shallow level only
                for sub in list(child.iterdir())[:50]:
                    q=sub/name if sub.is_dir() else None
                    if q is not None and q.exists(): out.append(q)
    return out

def load_backbone():
    model=resnet18(weights=None); base=model.state_dict()
    for p in quick_checkpoint_candidates():
        try:
            raw=torch.load(p,map_location='cpu',weights_only=True)
            if isinstance(raw,dict) and 'state_dict' in raw: raw=raw['state_dict']
            clean={}
            for k,v in raw.items():
                if not torch.is_tensor(v):continue
                kk=k
                for prefix in ['module.','model.','backbone.','encoder.']:
                    if kk.startswith(prefix):kk=kk[len(prefix):]
                if kk in base and tuple(v.shape)==tuple(base[kk].shape):clean[kk]=v
            coverage=len(clean)/len(base)
            if coverage>=.90:
                model.load_state_dict(clean,strict=False)
                return model,'imagenet_pretrained',str(p),coverage
        except Exception: pass
    return model,'report_guided_random_init',None,0.0

backbone,INIT_MODE,PRETRAIN_PATH,PRETRAIN_COVERAGE=load_backbone()
if INIT_MODE=='imagenet_pretrained':
    INPUT_MEAN=(0.485,0.456,0.406); INPUT_STD=(0.229,0.224,0.225); ENCODER_LR=ENCODER_LR_PRETRAINED
else:
    ENCODER_LR=ENCODER_LR_RANDOM
print('Encoder init:',INIT_MODE,'| checkpoint:',PRETRAIN_PATH,'| coverage:',round(PRETRAIN_COVERAGE,3))

## 13. Target-aware MIL + report-semantic auxiliary head

In [ ]:
class ReportGuidedMIL(nn.Module):
    def __init__(self,backbone,n_targets,feature_dim=256,report_dim=REPORT_DIM):
        super().__init__(); in_features=backbone.fc.in_features; backbone.fc=nn.Identity(); self.encoder=backbone
        self.proj=nn.Sequential(nn.Linear(in_features,feature_dim),nn.LayerNorm(feature_dim),nn.GELU())
        self.plane_emb=nn.Embedding(len(PLANES),feature_dim); self.pos_emb=nn.Embedding(TRIPLETS_PER_PLANE,feature_dim)
        self.queries=nn.Parameter(torch.randn(n_targets,feature_dim)*.02); self.weights=nn.Parameter(torch.randn(n_targets,feature_dim)*.02); self.bias=nn.Parameter(torch.zeros(n_targets))
        self.report_head=nn.Sequential(nn.Linear(feature_dim,feature_dim),nn.GELU(),nn.Linear(feature_dim,report_dim))
        self.dropout=nn.Dropout(.15); self.scale=feature_dim**-.5
    def forward(self,x,token_mask,plane_id,position_id):
        b,n,c,h,w=x.shape
        flat=x.reshape(b*n,c,h,w).contiguous(memory_format=torch.channels_last)
        feat=self.encoder(flat).reshape(b,n,-1); feat=self.proj(feat)
        feat=feat+self.plane_emb(plane_id)+self.pos_emb(position_id); feat=self.dropout(feat)
        attn_logits=torch.einsum('bnf,tf->bnt',feat,self.queries)*self.scale
        attn_logits=attn_logits.masked_fill(token_mask.unsqueeze(-1)<=0,-1e4)
        attn=torch.softmax(attn_logits,dim=1)
        pooled=torch.einsum('bnt,bnf->btf',attn,feat)
        logits=(pooled*self.weights.unsqueeze(0)).sum(-1)+self.bias
        denom=token_mask.sum(1,keepdim=True).clamp_min(1.0)
        global_feat=(feat*token_mask.unsqueeze(-1)).sum(1)/denom
        report_pred=self.report_head(global_feat)
        return logits,attn,report_pred

model=ReportGuidedMIL(backbone,len(TARGETS)).to(DEVICE)
model.encoder.to(memory_format=torch.channels_last)
print('Trainable parameters:',f'{sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')

## 14. Loaders, target-aware oversampling and finite-safe losses

In [ ]:
train_ds=KneeDataset(train_part,TARGETS,True,REPORT_EMBED)
val_ds=KneeDataset(val_part,TARGETS,False,REPORT_EMBED)
test_ds=KneeDataset(test_df,None,False,{})

Y=train_part[TARGETS].to_numpy(np.float32)
pos=np.nansum(Y==1,axis=0); neg=np.nansum(Y==0,axis=0)
raw_ratio=np.divide(neg,np.maximum(pos,1)); pos_weight=torch.tensor(np.clip(np.sqrt(np.maximum(raw_ratio,1.0)),1.0,MAX_POS_WEIGHT),dtype=torch.float32,device=DEVICE)

# Each positive rare target contributes inverse-sqrt frequency to the study sampling weight.
boost=1.0/np.sqrt(np.maximum(pos,1.0)); boost=boost/np.median(boost)
y01=np.nan_to_num(Y,nan=0.0)
sample_weights=1.0+(y01*boost[None,:]).sum(1)
sample_weights=np.clip(sample_weights,1.0,8.0)
sampler=WeightedRandomSampler(torch.as_tensor(sample_weights,dtype=torch.double),num_samples=len(train_ds),replacement=True)

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,sampler=sampler,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)

# Warm-up loader uses a bounded train-only subset and no target oversampling.
warm_frame=train_part.sample(n=min(WARMUP_MAX_STUDIES,len(train_part)),random_state=SEED).reset_index(drop=True)
warm_ds=KneeDataset(warm_frame,TARGETS,True,REPORT_EMBED)
warm_loader=DataLoader(warm_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=NUM_WORKERS>0)

def supervised_loss(logits,targets):
    valid=torch.isfinite(targets); y=torch.nan_to_num(targets,nan=0.0)
    loss=F.binary_cross_entropy_with_logits(logits,y,pos_weight=pos_weight,reduction='none')
    vf=valid.float(); counts=vf.sum(0); per=(loss*vf).sum(0)/counts.clamp_min(1.0); good=counts>0
    out=per[good].mean()
    if not torch.isfinite(out): raise FloatingPointError('Non-finite supervised loss.')
    return out

def semantic_loss(pred,target):
    good=torch.linalg.vector_norm(target,dim=1)>1e-6
    if not good.any(): return pred.sum()*0.0
    p=F.normalize(pred[good].float(),dim=1,eps=1e-6); t=F.normalize(target[good].float(),dim=1,eps=1e-6)
    out=(1.0-(p*t).sum(1)).mean()
    if not torch.isfinite(out): raise FloatingPointError('Non-finite report-semantic loss.')
    return out

def macro_auc(y,p):
    out={}
    for j,t in enumerate(TARGETS):
        mask=np.isfinite(y[:,j]); yt=y[mask,j]; yp=p[mask,j]
        out[t]=np.nan if len(np.unique(yt))<2 else roc_auc_score(yt,yp)
    return float(np.nanmean(list(out.values()))),out

display(pd.DataFrame({'positive':pos.astype(int),'negative':neg.astype(int),'pos_weight':pos_weight.cpu().numpy()},index=TARGETS).round(3))

## 15. One-epoch report-semantic warm-up

When no ImageNet checkpoint is available this is the critical representation step: the MRI encoder is forced to predict dense semantic structure extracted from paired radiology reports before target fine-tuning.

In [ ]:
# Freeze early encoder blocks during the short semantic warm-up for speed and stability.
for name,p in model.encoder.named_parameters():
    p.requires_grad = name.startswith('layer4')
warm_params=[p for p in model.parameters() if p.requires_grad]
warm_opt=torch.optim.AdamW(warm_params,lr=3e-4,weight_decay=WEIGHT_DECAY)
warm_scaler=torch.amp.GradScaler('cuda',enabled=AMP_ENABLED)

for epoch in range(REPORT_WARMUP_EPOCHS):
    model.train(); total=0.0; seen=0
    for batch in warm_loader:
        x=batch['image'].to(DEVICE,non_blocking=True); mask=batch['token_mask'].to(DEVICE,non_blocking=True); plane=batch['plane_id'].to(DEVICE,non_blocking=True); posid=batch['position_id'].to(DEVICE,non_blocking=True); rz=batch['report_embed'].to(DEVICE,non_blocking=True)
        if not torch.isfinite(x).all(): raise FloatingPointError('Non-finite MRI tensor after sanitization.')
        warm_opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=AMP_ENABLED):
            _,_,rp=model(x,mask,plane,posid); loss=semantic_loss(rp,rz)
        warm_scaler.scale(loss).backward(); warm_scaler.unscale_(warm_opt); torch.nn.utils.clip_grad_norm_(warm_params,3.0); warm_scaler.step(warm_opt); warm_scaler.update()
        total += float(loss.item())*x.size(0); seen += x.size(0)
    print(f'Report warm-up epoch {epoch+1}/{REPORT_WARMUP_EPOCHS} | semantic loss {total/max(seen,1):.5f} | elapsed {elapsed_min():.1f} min')

# Unfreeze entire encoder for supervised fine-tuning.
for p in model.encoder.parameters(): p.requires_grad=True
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 16. Supervised GPU fine-tuning with wall-clock guard

In [ ]:
encoder_params=list(model.encoder.parameters())
head_params=[p for n,p in model.named_parameters() if not n.startswith('encoder.')]
opt=torch.optim.AdamW([{'params':encoder_params,'lr':ENCODER_LR},{'params':head_params,'lr':HEAD_LR}],weight_decay=WEIGHT_DECAY)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=SUPERVISED_EPOCHS)
scaler=torch.amp.GradScaler('cuda',enabled=AMP_ENABLED)

best_auc=-np.inf; best_state=None; best_epoch=None; stale=0; history=[]
for epoch in range(1,SUPERVISED_EPOCHS+1):
    if time.time()-START_TIME > TRAINING_CUTOFF_SECONDS:
        print('⏱️ Runtime guard: stopping training to preserve inference/export budget.'); break
    model.train(); running=0.0; seen=0
    for batch in train_loader:
        x=batch['image'].to(DEVICE,non_blocking=True); mask=batch['token_mask'].to(DEVICE,non_blocking=True); plane=batch['plane_id'].to(DEVICE,non_blocking=True); posid=batch['position_id'].to(DEVICE,non_blocking=True); y=batch['target'].to(DEVICE,non_blocking=True); rz=batch['report_embed'].to(DEVICE,non_blocking=True)
        if not torch.isfinite(x).all(): raise FloatingPointError(f'Non-finite MRI tensor. UIDs: {batch["uid"][:3]}')
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=AMP_ENABLED):
            logits,_,rp=model(x,mask,plane,posid)
            if not torch.isfinite(logits).all(): raise FloatingPointError(f'Non-finite logits. UIDs: {batch["uid"][:3]}')
            cls=supervised_loss(logits,y); sem=semantic_loss(rp,rz); loss=cls+REPORT_AUX_WEIGHT*sem
        if not torch.isfinite(loss): raise FloatingPointError('Non-finite total loss.')
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),3.0); scaler.step(opt); scaler.update()
        running += float(loss.item())*x.size(0); seen += x.size(0)

    model.eval(); vp=[]; vy=[]
    with torch.no_grad():
        for batch in val_loader:
            x=batch['image'].to(DEVICE,non_blocking=True); mask=batch['token_mask'].to(DEVICE,non_blocking=True); plane=batch['plane_id'].to(DEVICE,non_blocking=True); posid=batch['position_id'].to(DEVICE,non_blocking=True)
            with torch.amp.autocast('cuda',enabled=AMP_ENABLED): logits,_,_=model(x,mask,plane,posid)
            vp.append(torch.sigmoid(logits).float().cpu().numpy()); vy.append(batch['target'].numpy())
    vp=np.concatenate(vp); vy=np.concatenate(vy); auc,per=macro_auc(vy,vp)
    row={'epoch':epoch,'train_total_loss':running/max(seen,1),'val_macro_auc':auc,'elapsed_min':elapsed_min()}; row.update({f'auc_{k}':v for k,v in per.items()}); history.append(row)
    print(f'Epoch {epoch:02d}/{SUPERVISED_EPOCHS} | loss {row["train_total_loss"]:.5f} | val macro AUC {auc:.5f} | elapsed {elapsed_min():.1f} min')
    if auc > best_auc + 1e-5:
        best_auc=auc; best_epoch=epoch; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
    else: stale += 1
    sched.step()
    if epoch>=2 and stale>=EARLY_STOPPING_PATIENCE:
        print('Early stopping: validation macro AUC did not improve.'); break

if best_state is None: raise RuntimeError('No valid checkpoint was produced.')
history_df=pd.DataFrame(history); history_df.to_csv('history.csv',index=False)
print('Best epoch:',best_epoch,'| best validation macro AUC:',round(best_auc,6))

## 17. Validation diagnostics

In [ ]:
model.load_state_dict(best_state); model.to(DEVICE).eval()
vp=[]; vy=[]; va=[]
with torch.no_grad():
    for batch in val_loader:
        x=batch['image'].to(DEVICE,non_blocking=True); mask=batch['token_mask'].to(DEVICE,non_blocking=True); plane=batch['plane_id'].to(DEVICE,non_blocking=True); posid=batch['position_id'].to(DEVICE,non_blocking=True)
        with torch.amp.autocast('cuda',enabled=AMP_ENABLED): logits,attn,_=model(x,mask,plane,posid)
        vp.append(torch.sigmoid(logits).float().cpu().numpy()); vy.append(batch['target'].numpy()); va.append(attn.float().cpu().numpy())
vp=np.concatenate(vp); vy=np.concatenate(vy); va=np.concatenate(va)
best_macro,target_auc=macro_auc(vy,vp); auc_series=pd.Series(target_auc).sort_values()

fig,ax=plt.subplots(figsize=(10,6)); auc_series.plot(kind='barh',ax=ax); ax.axvline(.5,linestyle=':'); ax.axvline(best_macro,linestyle='--',label=f'macro={best_macro:.4f}'); ax.set_xlim(0,1); ax.set_title('Per-target validation ROC AUC'); ax.legend(); ax.grid(axis='x',alpha=.25); plt.tight_layout(); plt.show()
display(auc_series.to_frame('validation_auc').round(5))

fig,ax=plt.subplots(figsize=(8,4)); ax.plot(history_df.epoch,history_df.val_macro_auc,marker='o'); ax.axhline(.5,linestyle=':'); ax.axhline(.62,linestyle='--'); ax.set_title('Validation macro ROC AUC'); ax.set_xlabel('Epoch'); ax.grid(alpha=.25); plt.tight_layout(); plt.show()

plane_attn=np.zeros((len(TARGETS),len(PLANES)))
for p_idx in range(len(PLANES)):
    ids=list(range(p_idx*TRIPLETS_PER_PLANE,(p_idx+1)*TRIPLETS_PER_PLANE)); plane_attn[:,p_idx]=va[:,ids,:].sum(1).mean(0)
fig,ax=plt.subplots(figsize=(7,7)); im=ax.imshow(plane_attn,aspect='auto'); ax.set_xticks(range(3)); ax.set_xticklabels(PLANES); ax.set_yticks(range(len(TARGETS))); ax.set_yticklabels(TARGETS); ax.set_title('Target-specific attention mass by MRI plane'); fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

if best_macro < .58: print('REJECT: validation remains too close to random.')
elif best_macro < .62: print('WEAK SIGNAL: above random, below promotion gate.')
elif best_macro < .68: print('PROMISING BASELINE: promote only after a second split confirms the gain.')
else: print('STRONG SINGLE-FOLD BASELINE: move to grouped multi-fold OOF.')

## 18. Raw test inference and submission

In [ ]:
test_probs=[]
with torch.no_grad():
    for batch in test_loader:
        x=batch['image'].to(DEVICE,non_blocking=True); mask=batch['token_mask'].to(DEVICE,non_blocking=True); plane=batch['plane_id'].to(DEVICE,non_blocking=True); posid=batch['position_id'].to(DEVICE,non_blocking=True)
        with torch.amp.autocast('cuda',enabled=AMP_ENABLED): logits,_,_=model(x,mask,plane,posid)
        test_probs.append(torch.sigmoid(logits).float().cpu().numpy())

test_probs=np.concatenate(test_probs)
pred=pd.DataFrame(test_probs,columns=TARGETS); pred[SUBMISSION_ID]=test_df[SUBMISSION_ID].astype(str).values
submission=sample_submission[[SUBMISSION_ID]].copy(); submission[SUBMISSION_ID]=submission[SUBMISSION_ID].astype(str); submission=submission.merge(pred,on=SUBMISSION_ID,how='left')
if submission[TARGETS].isna().any().any(): raise RuntimeError('Missing test predictions.')
submission.to_csv('submission.csv',index=False)

torch.save({'model_state_dict':best_state,'targets':TARGETS,'validation_macro_auc':best_macro,'validation_target_auc':target_auc,'init_mode':INIT_MODE,'pretrained_path':PRETRAIN_PATH,'config':{'img_size':IMG_SIZE,'triplets_per_plane':TRIPLETS_PER_PLANE,'report_dim':REPORT_DIM,'report_warmup_epochs':REPORT_WARMUP_EPOCHS,'supervised_epochs':SUPERVISED_EPOCHS,'batch_size':BATCH_SIZE,'seed':SEED}},'rsna_knee_report_guided_v3.pth')
print('submission.csv:',submission.shape)
print(f'Total elapsed: {elapsed_min():.1f} min')
if time.time()-START_TIME > MAX_RUNTIME_SECONDS: print('⚠️ Runtime exceeded the requested two-hour target on this environment.')
display(submission.head())

## 19. Decision

This notebook is a **challenger**, not a presumed winner. The report-guided warm-up has a plausible causal mechanism: it injects dense semantic supervision from the paired radiology reports into a train-only image representation, then tests whether that representation improves raw MRI validation AUC.

**Keep only measured gains.** If the single-fold macro AUC does not materially exceed the previous ~0.51 regime, the next move should be an MRI-native pretrained encoder or target-specific anatomical localization—not more epochs of the same model.

### Strongest counterargument

TF-IDF/SVD report semantics may encode broad radiology language that is only weakly recoverable from six sampled MRI triplets. If validation does not improve, that falsifies this particular privileged-information transfer mechanism; it does not imply that multimodal learning itself is useless.